In [ ]:
!pip install transformers==4.57.6

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
device = "cuda"
model_name = "Qwen/Qwen2.5-Coder-0.5B-Instruct" 

In [ ]:
from typing import Any


tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, padding_side="right")
# tokenizer.pad_token = tokenizer.eos_token = "<|endoftext|>"
# --- Dataset ---
dataset = load_dataset("json", data_files="/kaggle/input/code-completion-train-data/psm-lineblock-1024-05.jsonl")
def tokenize(example):
    return tokenizer(example["text"], truncation=False, padding=False, max_length=1024)
dataset = dataset.map(tokenize, batched=True, remove_columns=["text"])
# Used to pad tokens, since each sample have different length
class CustomCausalCollator:
    def __init__(self, tokenizer) -> None:
        self.tokenizer = tokenizer
    def __call__(self, features) -> Any:
        batch = self.tokenizer.pad(
            features,
            return_tensors="pt"
        )
        labels = batch["input_ids"].clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        batch["labels"] = labels
        return batch
data_collator = CustomCausalCollator(tokenizer)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
).to(device, dtype=torch.bfloat16)

# --- LoRA config ---
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_cfg).to(device, dtype=torch.bfloat16)

# --- Training ---
args = TrainingArguments(
    output_dir="qwen25-05i-spm-lineblock",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    num_train_epochs=1,
    bf16=True,               # enables CUDA bfloat16
    logging_steps=256,
    save_strategy="steps",
    save_steps=256, # 4096 samples
    save_total_limit=100,      
    save_safetensors=True,   # ✅ use safetensors format (recommended)
    optim="adamw_torch",
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=dataset["train"], data_collator=data_collator)
trainer.train()